In [ ]:
import pandas as pd
import miceforest as mf
import matplotlib.pyplot as plt
import numpy as np
import os
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

In [ ]:
os.chdir(r'T:\lab_research\RES-Folder-UPOD\ODIN-UC4\E_ResearchData\2_ResearchData\20240805')

In [ ]:
df_lab = pd.read_parquet('lab_20240805.parquet')
df_lab.studyId_0831 = df_lab.studyId_0831.astype(int)

Let's remove uneuseful columns

In [ ]:
df_lab = df_lab.drop(['identifier_value', 'basedOn_ProcedureRequest_value',
                      'LabID', 'category3_display_original', 'category4_code_original',
                      'code_code_original', 'code2_code_original',
                      'code2_display_original', 'code3_display_original',
                      'interpretation_display',
                      'code3_system_original', 'code4_display_original', 'comment',
                      'status_display_original',
                      'valueQuantity_comparator', 'valueQuantity_unit',
                      'valueQuantity_unit_original', 'valueString',
                      'Practitioner_Rol', 'gender', 'organization_Organization_value',
                      'specialty_specialtyAGB_display', 'index_date'], axis='columns')

Storing the reference range of values per substance

In [ ]:
substances = pd.read_excel(
    r'T:\lab_research\RES-Folder-UPOD\ODIN-UC4\G_Output\2_Data\excel\measurements_ranges.xlsx')
substances = substances.set_index('substance')


substances.loc['Kreatinine']['unit']

Pivoting on the chemical sub

In [ ]:
df_lab

In [ ]:
df_lab.pivot(columns='code_display_original', values='valueQuantity_value').notna().sum()

---
To be interpolated data

In [ ]:
tb_interp = df_lab\
    .query('code_display_original in ["Kreatinine", "eGFR (CKD-EPI)","Natrium","Kalium","Hemoglobine"]')
tb_interp

In [ ]:
tb_interp_p = tb_interp.pivot(columns='code_display_original', values='valueQuantity_value')
tb_interp_p['studyId_0831'] = tb_interp.studyId_0831
tb_interp_p

In [ ]:
def interp_and_wavg(x : pd.Series):
    if x.count() == 1:
        return x.fillna(0).sum()
    elif x.count() > 1:
        x = x.interpolate('slinear').fillna(0) #Interpolation was failing here
        weights = np.linspace(0,1,len(x))
        return np.average(x,weights=weights)
    else:
        return np.nan

In [ ]:
tb_interp_p_gr = tb_interp_p.groupby('studyId_0831').agg(interp_and_wavg)
tb_interp_p_gr

In [ ]:
tb_interp_p_gr.isna().sum(axis='columns').value_counts()

In [ ]:
tb_interp_p.notna().sum()

---
To be imputed data

In [ ]:
tb_imputed = df_lab\
    .query('code_display_original not in ["Kreatinine", "eGFR (CKD-EPI)","Natrium","Kalium","Hemoglobine"]')
tb_imputed

In [ ]:
tb_imputed_p = tb_imputed.pivot(
    columns='code_display_original', values='valueQuantity_value')
tb_imputed_p['studyId_0831'] = tb_imputed.studyId_0831
tb_imputed_p = tb_imputed_p.reset_index(drop=True)
tb_imputed_p

In [ ]:
ker = mf.ImputationKernel(
    tb_imputed_p,
    num_datasets=1,
    save_all_iterations_data=True,
    mean_match_strategy='shap',
    random_state=42)

ker.mice(iterations=5,
         n_estimators=100,
         boosting='gbdt',
         min_sum_hessian_in_leaf=0.01)

In [ ]:
tb_imp_p_imp = ker.complete_data()
tb_imp_p_imp

In [ ]:
ker.plot_imputed_distributions()